# **Sequential Decision-Making and $Q$-Learning**

Imagine a small delivery robot moving through a building. It starts in a hallway, chooses where to move, receives feedback, and must learn to reach a charging station while avoiding blocked areas. It is not given the route in advance; it learns from repeated attempts.

This is a simplified model of problems such as robot navigation, game-playing, and adaptive resource control. We will first build the mathematical model, then implement a small grid example in Python using only lists, dictionaries, and loops.

## **1. Model the robot's decisions mathematically**

At each time step $t$, the robot observes where it is, chooses a move, receives a reward, and arrives at a new location:

$$s_t \xrightarrow{a_t,\ r_{t+1}} s_{t+1}.$$

A **Markov decision process (MDP)** is described by:

$$\mathcal{M} = \langle \mathcal{S}, \mathcal{A}, P, R, \gamma \rangle.$$

- $\mathcal{S}$ is the set of possible **states**. Here, a state is the robot's grid location.
- $\mathcal{A}$ is the set of possible **actions**, such as moving up, down, left, or right.
- $P(s' \mid s,a)$ describes the probability of reaching state $s'$ after taking action $a$ in state $s$. In our small grid, moves are deterministic unless blocked by a wall or boundary.
- $R(s,a,s')$ is the immediate **reward** for that move. A step can have a small cost, while reaching the destination earns a larger reward.
- $\gamma \in [0,1]$ is the **discount factor**: it controls how much future rewards count compared with immediate rewards.

A **policy** $\pi(a \mid s)$ describes how the robot chooses an action in each state. The goal is to learn a policy that produces high total reward.

The Markov property means that the current state contains enough information to predict the next state and reward; we do not need the entire history of moves.

## **2. From rewards to the $Q$-learning update**

The robot should prefer actions that lead to larger rewards over time, not just moves with a good immediate reward. The discounted return from time $t$ is:

$$G_t = r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \cdots.$$

The action-value $Q(s,a)$ estimates the return the robot can expect if it takes action $a$ in state $s$ and then continues following its learned behavior. We store one estimate for every state-action pair.

After trying an action, the robot has:

- its old estimate $Q(s_t,a_t)$,
- the reward $r_{t+1}$ it just received,
- and the next state $s_{t+1}$.

For a non-terminal next state, the one-step target is the reward plus the best estimated future value:

$$y_t = r_{t+1} + \gamma \max_{a'} Q(s_{t+1},a').$$

The temporal-difference error is the difference between this target and the old estimate:

$$\delta_t = y_t - Q(s_t,a_t).$$

The $Q$-learning update moves the old estimate partway toward the target:

$$Q(s_t,a_t) \leftarrow Q(s_t,a_t) + \alpha\,\delta_t,$$

where $\alpha \in (0,1]$ is the **learning rate**. Combining the two equations gives:

$$Q(s_t,a_t) \leftarrow Q(s_t,a_t) + \alpha\left[r_{t+1} + \gamma\max_{a'}Q(s_{t+1},a') - Q(s_t,a_t)\right].$$

If the robot has reached a terminal state, there is no future reward to estimate, so the target is simply $y_t=r_{t+1}$.

### **2.1 One numerical update**

Suppose a move costs $-1$, the current estimate is $Q(s,a)=0$, all next-state estimates are $0$, and $\alpha=0.2$. Then:

$$y = -1 + 0.95(0) = -1, \qquad \delta = -1 - 0 = -1,$$

so the updated estimate is:

$$Q(s,a) = 0 + 0.2(-1) = -0.2.$$

The robot has learned that this move was slightly costly. Repeated experience will make the estimates reflect which routes reach the goal efficiently.

### **2.2 Exploration and exploitation**

The robot needs to try actions to discover what works (**exploration**) and also use actions it already believes are good (**exploitation**). An $\epsilon$-greedy rule does this:

- with probability $\epsilon$, choose a random action;
- with probability $1-\epsilon$, choose the action with the largest current $Q(s,a)$.

A large $\epsilon$ explores more. A small $\epsilon$ exploits current knowledge more.

## **3. Translate the scenario into a small grid**

We will use a $4 \times 4$ grid. A state is a pair $(row, column)$, with both coordinates starting at 0. The robot starts at the upper-left cell and must reach the lower-right goal cell. Two cells are blocked.

The grid is small enough to inspect directly. `S` is the start, `G` is the goal, `#` is a blocked cell, and `.` is an open cell.

In [1]:
import random

SIZE = 4
START = (0, 0)
GOAL = (3, 3)
WALLS = {(1, 1), (2, 1)}

# Each action is a change in (row, column).
ACTIONS = [(0, 1), (1, 0), (0, -1), (-1, 0)]
ACTION_NAMES = ["right", "down", "left", "up"]

STATES = [
    (row, column)
    for row in range(SIZE)
    for column in range(SIZE)
    if (row, column) not in WALLS
]

for row in range(SIZE):
    cells = []
    for column in range(SIZE):
        position = (row, column)
        if position == START:
            cells.append("S")
        elif position == GOAL:
            cells.append("G")
        elif position in WALLS:
            cells.append("#")
        else:
            cells.append(".")
    print(" ".join(cells))

S . . .
. # . .
. # . .
. . . G


### **3.1 Define what one move does**

An action changes the row and column. For example, moving right adds $(0,1)$ to the current position. The transition function below applies the move, keeps the robot inside the grid, and prevents it from entering a wall.

For feedback, each ordinary move has reward $-1$ and reaching the goal has reward $+10$. The small step cost encourages shorter routes.

In [2]:
def step(state, action):
    row, column = state
    row_change, column_change = action

    candidate = (
        max(0, min(SIZE - 1, row + row_change)),
        max(0, min(SIZE - 1, column + column_change)),
    )

    # A wall blocks movement, so the robot stays in its current cell.
    next_state = state if candidate in WALLS else candidate
    done = next_state == GOAL
    reward = 10 if done else -1
    return next_state, reward, done

### **3.2 Store the $Q$ estimates**

Each open cell is a state. For every state, we store one number per action. At the beginning, all estimates are zero because the robot has not learned anything yet.

The list order matches `ACTIONS`: right, down, left, up. The helper `choose_action` implements the $\epsilon$-greedy rule from the mathematical section.

In [3]:
q_values = {state: [0.0] * len(ACTIONS) for state in STATES}


def choose_action(state, epsilon):
    if random.random() < epsilon:
        return random.randrange(len(ACTIONS))
    return max(range(len(ACTIONS)), key=lambda index: q_values[state][index])


print("Number of states:", len(q_values))
print("Actions per state:", len(ACTIONS))

Number of states: 14
Actions per state: 4


## **4. Train the robot with Q-learning**

Each training step follows the same sequence as the update equation:

1. Choose an action using $\epsilon$-greedy exploration.
2. Observe the next state, reward, and whether the goal was reached.
3. Compute the target $y_t$.
4. Compute the temporal-difference error $\delta_t = y_t - Q(s_t,a_t)$.
5. Update only the value for the state-action pair that was just tried.

The code below uses $\alpha=0.2$, $\gamma=0.95$, and $\epsilon=0.2$. A fixed random seed makes the classroom run reproducible.

In [4]:
random.seed(7)

alpha = 0.2
gamma = 0.95
epsilon = 0.2
number_of_episodes = 1000
max_steps_per_episode = 40

for episode in range(number_of_episodes):
    state = START

    for step_number in range(max_steps_per_episode):
        action_index = choose_action(state, epsilon)
        action = ACTIONS[action_index]
        next_state, reward, done = step(state, action)

        if done:
            target = reward
        else:
            target = reward + gamma * max(q_values[next_state])

        old_estimate = q_values[state][action_index]
        td_error = target - old_estimate
        q_values[state][action_index] = old_estimate + alpha * td_error

        state = next_state
        if done:
            break

print("Training complete.")

Training complete.


## **5. Test the learned policy**

Training explores and updates the table. To test what was learned, we now stop exploration and always choose the action with the largest stored $Q$ value. The route is a sequence of grid coordinates.

In [5]:
state = START
route = [state]

for _ in range(SIZE * SIZE * 2):
    if state == GOAL:
        break

    action_index = max(range(len(ACTIONS)), key=lambda index: q_values[state][index])
    next_state, reward, done = step(state, ACTIONS[action_index])
    state = next_state
    route.append(state)

    if done:
        break

reached_goal = route[-1] == GOAL
print("Greedy route:", route)
print("Reached goal:", reached_goal)
assert reached_goal
assert not any(position in WALLS for position in route)

route_cells = set(route)
for row in range(SIZE):
    cells = []
    for column in range(SIZE):
        position = (row, column)
        if position == START:
            cells.append("S")
        elif position == GOAL:
            cells.append("G")
        elif position in WALLS:
            cells.append("#")
        elif position in route_cells:
            cells.append("*")
        else:
            cells.append(".")
    print(" ".join(cells))

Greedy route: [(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3), (3, 3)]
Reached goal: True
S * * *
. # . *
. # . *
. . . G


### **Interpret the result**

The printed route is generated from the learned $Q$ table; it was not written into the program as a sequence of directions. `S` marks the start, `G` the goal, `#` a wall, and `*` a cell visited by the greedy policy.

The step reward is $-1$, so each extra move reduces the return. Reaching the goal gives $+10$. During training, $\epsilon=0.2$ means the robot usually follows its current best estimate but sometimes tries a random action.

**Check your understanding:**

1. Why does the update target equal only the reward when `done` is true?
2. Why do we reset the Q table before comparing two experiments?
3. What might happen if $\epsilon=0$ before the robot has discovered the goal?

## **6. Lab: run a controlled experiment**

The robot can already navigate around walls. Now investigate how its learning changes when you alter one design choice at a time.

### **Tasks**

1. Run training with $\epsilon=0.0$, $0.2$, and $0.5$. Before each run, reset every Q estimate to zero and keep the same rewards and episode count. Record whether the greedy route reaches the goal.
2. Explain why a value of $\epsilon=0$ can make learning unreliable when all initial Q estimates are equal.
3. Keep $\epsilon=0.2$ and compare a step reward of $-1$ with $-2$. Does the agent prefer a shorter route? What other effect could the changed reward have on learning?
4. Add one more wall, but keep at least one route to the goal. Update the state list and Q table, retrain, and draw the resulting route.

For a fair comparison, change one parameter at a time and reset the Q table before retraining.

In [ ]:
# Experiment: add one wall that still leaves a route to the goal.
WALLS.add((1, 2))

# Rebuild the state space and Q table after changing the environment.
STATES = [
    (row, column)
    for row in range(SIZE)
    for column in range(SIZE)
    if (row, column) not in WALLS
]
q_values = {state: [0.0] * len(ACTIONS) for state in STATES}

# TODO: choose an epsilon value, then repeat the training loop from Section 4.
# Run the greedy policy again and verify that it reaches GOAL without entering a wall.

## **7. Exit ticket and further reading**

Submit the transition function, the training loop, and the greedy route produced by your agent. In a short explanation, define $s$, $a$, $r$, $\gamma$, $\alpha$, and $\epsilon$, then describe how one observed transition changes one entry in the Q table.

**Extension:** make one action occasionally move in a random direction. Explain how this changes the transition model $P(s' \mid s,a)$ and why the agent may need more training.

### **AIMA reading guide**

Russell and Norvig, *Artificial Intelligence: A Modern Approach*, 4th edition:

- **Chapter 17, Section 17.1, “Sequential Decision Problems”**: states, actions, rewards, policies, and discounted utility over time. Read **17.1.1, “Utilities over Time,”** for the role of $\gamma$, and **17.1.4, “Representing MDPs,”** for the MDP model.
- **Chapter 22, Section 22.1, “Learning from Rewards”**: the basic reinforcement-learning setting and value estimates.
- **Chapter 22, Section 22.2.3, “Temporal-Difference Learning”**: the target and prediction-error idea behind the update.
- **Chapter 22, Section 22.3.1, “Exploration”**: why the agent must sometimes try actions it does not currently prefer.
- **Chapter 22, Section 22.3.3, “Temporal-Difference Q-learning”**: the Q-learning update rule used in this notebook.

The official [AIMA table of contents](https://aima.cs.berkeley.edu/contents.html) lists these sections for the 4th US edition.